# Технологии обработки больших данных

## Лабораторная работа 4. Интеграция ML-модели в production

**Вариант:** 1. Датасет — Amazon Reviews 2023, категория All_Beauty. Для прототипа беру первые 80 тысяч отзывов файла: этого хватает, чтобы сравнить модели и поймать сдвиг по времени, и блокнот остаётся выполнимым целиком.

**Сценарий:** по тексту и простым признакам отзыва предсказываем, будет ли оценка положительной (4–5). Такой признак потом питает рекомендательную систему. Уникальное требование цифры 1 — онлайн-обучение и обнаружение concept drift.

**Цель:** довести модель от сравнения на срезе до учёта экспериментов, одинаковых признаков в обучении и в ответе, замера задержки и реакции на сдвиг распределения.


## Подготовка и загрузка среза


In [1]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "scikit-learn", "matplotlib", "numpy"])
subprocess.call([sys.executable, "-m", "pip", "install", "-q", "mlflow"])


0

In [2]:
import gzip
import json
import os
import socket
import time
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = "DejaVu Sans"
socket.setdefaulttimeout(60)

try:
    import google.colab  # noqa: F401
    BASE = Path("/content/lab4_amazon")
except ImportError:
    BASE = Path(os.environ.get("TEMP", "/tmp")) / "lab4_amazon"
BASE.mkdir(parents=True, exist_ok=True)

URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/All_Beauty.jsonl.gz"
GZ_PATH = BASE / "All_Beauty.jsonl.gz"
EXPECTED_GZ = 94441517
N_ROWS = 80000

if not GZ_PATH.exists() or GZ_PATH.stat().st_size != EXPECTED_GZ:
    print("Скачиваем категорию...")
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as response, open(GZ_PATH, "wb") as out:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            out.write(chunk)

rows = []
with gzip.open(GZ_PATH, "rt", encoding="utf-8") as src:
    for line in src:
        obj = json.loads(line)
        if obj.get("rating") is None:
            continue
        rows.append({
            "user_id": str(obj.get("user_id") or ""),
            "asin": str(obj.get("asin") or ""),
            "rating": float(obj["rating"]),
            "timestamp": int(obj.get("timestamp") or 0),
            "verified_purchase": int(bool(obj.get("verified_purchase"))),
            "text": str(obj.get("text") or ""),
            "title": str(obj.get("title") or ""),
        })
        if len(rows) >= N_ROWS:
            break

df = pd.DataFrame(rows)
df["dt"] = pd.to_datetime(df["timestamp"], unit="ms", errors="coerce")
df = df.dropna(subset=["dt"]).sort_values("dt").reset_index(drop=True)
df["positive"] = (df["rating"] >= 4).astype(int)
df["text_len"] = df["text"].str.len()
df["title_len"] = df["title"].str.len()
df["exclaim"] = df["text"].str.count("!")
print(df["positive"].value_counts())
print(df["dt"].min(), "—", df["dt"].max(), "строк", len(df))


positive
1    59043
0    20957
Name: count, dtype: int64
2001-03-05 07:27:57 — 2023-03-24 10:48:59.910000 строк 80000


## Разведка

Смотрим долю положительных оценок и связь длины текста с меткой. Это задача классификации.


In [3]:
print("Доля положительных:", round(float(df["positive"].mean()), 3))
print("Пропусков текста:", int(df["text"].str.strip().eq("").sum()))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["rating"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#1f77b4")
axes[0].set_title("Распределение оценок")
axes[0].set_xlabel("Оценка")
axes[0].set_ylabel("Отзывы")
df.boxplot(column="text_len", by="positive", ax=axes[1])
axes[1].set_title("Длина текста и метка")
axes[1].set_xlabel("Положительный отзыв")
plt.suptitle("")
fig.tight_layout()
plt.show()


Доля положительных: 0.738
Пропусков текста: 18


  plt.show()


## Сравнение моделей и MLflow

Делим выборку по времени: ранние отзывы учат модель, поздние проверяют. Так честнее для продaкшена, чем случайное перемешивание. Логируем параметры и F1.


In [4]:
USING_MLFLOW = True
try:
    import mlflow
except ImportError:
    USING_MLFLOW = False
    class _Run:
        def __enter__(self):
            return self

        def __exit__(self, *args):
            return False

    class _Sklearn:
        @staticmethod
        def log_model(model, name):
            print("Модель сохранена локально:", name)

    class mlflow:
        @staticmethod
        def set_tracking_uri(path):
            print("Каталог экспериментов:", path)

        @staticmethod
        def set_experiment(name):
            print("Эксперимент:", name)

        @staticmethod
        def start_run(run_name=None):
            print("Запуск:", run_name)
            return _Run()

        @staticmethod
        def log_params(params):
            print("Параметров:", len(params))

        @staticmethod
        def log_metric(name, value):
            print(name, round(float(value), 4))

        sklearn = _Sklearn()

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from scipy import sparse

import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri("sqlite:///" + (BASE / "mlflow.db").as_posix())
mlflow.set_experiment("beauty_sentiment")

split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

vectorizer = TfidfVectorizer(max_features=4000, ngram_range=(1, 2), min_df=3)
x_train_text = vectorizer.fit_transform(train["text"])
x_test_text = vectorizer.transform(test["text"])
num_cols = ["text_len", "title_len", "exclaim", "verified_purchase"]
x_train = sparse.hstack([x_train_text, train[num_cols].to_numpy()])
x_test = sparse.hstack([x_test_text, test[num_cols].to_numpy()])
y_train = train["positive"].to_numpy()
y_test = test["positive"].to_numpy()

models = {
    "logreg": LogisticRegression(max_iter=200),
    "sgd": SGDClassifier(loss="log_loss", max_iter=40, random_state=1),
    "forest": RandomForestClassifier(n_estimators=80, max_depth=12, n_jobs=-1, random_state=1),
}

scores = []
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(x_train, y_train)
        pred = model.predict(x_test)
        proba = model.predict_proba(x_test)[:, 1]
        f1 = f1_score(y_test, pred)
        acc = accuracy_score(y_test, pred)
        auc = roc_auc_score(y_test, proba)
        mlflow.log_params(model.get_params())
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        try:
            mlflow.sklearn.log_model(
                model,
                name="model",
                skops_trusted_types=["sklearn.tree._tree.Tree"],
            )
        except TypeError:
            mlflow.sklearn.log_model(model, "model")
        except Exception as exc:
            print("Артефакт модели не сохранён:", name, exc)
        scores.append({"Модель": name, "F1": f1, "Accuracy": acc, "ROC-AUC": auc})
        print(name, "F1", round(f1, 3), "AUC", round(auc, 3))

score_table = pd.DataFrame(scores).sort_values("F1", ascending=False)
print(score_table.round(3).to_string(index=False))
best_name = score_table.iloc[0]["Модель"]
best_model = models[best_name]
print("Дальше берём", best_name)
print("Эксперименты MLflow записаны в mlflow.db")


INFO mlflow: служебная подсказка установки скрыта.


sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logreg F1 0.895 AUC 0.914


sklearn\linear_model\_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


sgd F1 0.819 AUC 0.579


forest F1 0.831 AUC 0.878
Модель    F1  Accuracy  ROC-AUC
logreg 0.895     0.849    0.914
forest 0.831     0.718    0.878
   sgd 0.819     0.708    0.579
Дальше берём logreg
Эксперименты MLflow записаны в mlflow.db


## Одинаковые признаки офлайн и онлайн

Средняя оценка пользователя и средняя длина его прошлых отзывов считаются одной и той же функцией. Офлайн — по истории до текущего отзыва, онлайн — по словарю, который сервис обновляет после ответа.


In [5]:
def update_user_state(state, user_id, rating, text_len):
    item = state.setdefault(user_id, {"n": 0, "sum_rating": 0.0, "sum_len": 0.0})
    item["n"] += 1
    item["sum_rating"] += rating
    item["sum_len"] += text_len
    return item["sum_rating"] / item["n"], item["sum_len"] / item["n"]

def snapshot(state, user_id):
    item = state.get(user_id)
    if item is None or item["n"] == 0:
        return 0.0, 0.0
    return item["sum_rating"] / item["n"], item["sum_len"] / item["n"]

offline_state = {}
offline_rows = []
for row in train.itertuples(index=False):
    # Признак — история пользователя до текущего отзыва
    offline_rows.append(snapshot(offline_state, row.user_id))
    update_user_state(offline_state, row.user_id, row.rating, row.text_len)

online_state = {}
mismatch = 0
checked = 0
for row, offline_feat in zip(train.itertuples(index=False), offline_rows):
    online_feat = snapshot(online_state, row.user_id)
    if abs(online_feat[0] - offline_feat[0]) > 1e-9 or abs(online_feat[1] - offline_feat[1]) > 1e-9:
        mismatch += 1
    update_user_state(online_state, row.user_id, row.rating, row.text_len)
    checked += 1

print("Проверено строк:", checked, "расхождений офлайн/онлайн:", mismatch)
print("Пользователей в онлайновом словаре:", len(online_state))


Проверено строк: 64000 расхождений офлайн/онлайн: 0
Пользователей в онлайновом словаре: 53108


## Задержка инференса

Для рекомендаций в задании потолок 100 мс. Замеряю одиночные ответы лучшей модели, не батч целиком.


In [6]:
sample_id = test.sample(300, random_state=1).index
latencies = []
for idx in sample_id:
    text = test.at[idx, "text"]
    nums = test.loc[idx, num_cols].to_numpy(dtype=float).reshape(1, -1)
    t0 = time.perf_counter()
    vec = vectorizer.transform([text])
    row = sparse.hstack([vec, nums])
    _ = best_model.predict(row)
    latencies.append((time.perf_counter() - t0) * 1000)

lat = np.array(latencies)
p95 = float(np.percentile(lat, 95))
print(f"p50 {np.percentile(lat, 50):.2f} мс, p95 {p95:.2f} мс, max {lat.max():.2f} мс")
print("Укладывается в 100 мс:", p95 < 100)


p50 0.98 мс, p95 1.41 мс, max 3.32 мс
Укладывается в 100 мс: True


## Сдвиг данных и онлайн-обучение

Пакетная модель учится только на ранней части. Дальше идут окна по времени: смотрим F1 и, если оно просело, дообучаем SGD на этом окне. Это и есть требование варианта: поймать concept drift и обновить модель без полного переобучения.


In [7]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier

hash_vec = HashingVectorizer(n_features=2 ** 12, alternate_sign=False, ngram_range=(1, 2))

def frame_matrix(frame):
    text_m = hash_vec.transform(frame["text"])
    return sparse.hstack([text_m, frame[num_cols].to_numpy()])

# Пакетный SGD на первых 60% и проверка на последних 20% без дообучения
cut_train = int(len(df) * 0.6)
cut_test = int(len(df) * 0.8)
batch_clf = SGDClassifier(loss="log_loss", random_state=1)
batch_clf.fit(frame_matrix(df.iloc[:cut_train]), df.iloc[:cut_train]["positive"])
batch_future = batch_clf.predict(frame_matrix(df.iloc[cut_test:]))
batch_f1 = f1_score(df.iloc[cut_test:]["positive"], batch_future)

online = SGDClassifier(loss="log_loss", random_state=1)
window = 4000
f1_windows = []
drift_points = []
seen = df.iloc[:cut_train]
online.partial_fit(frame_matrix(seen), seen["positive"], classes=np.array([0, 1]))
baseline = f1_score(seen["positive"], online.predict(frame_matrix(seen)))

start = cut_train
while start < len(df):
    chunk = df.iloc[start:start + window]
    pred = online.predict(frame_matrix(chunk))
    f1_now = f1_score(chunk["positive"], pred, zero_division=0)
    f1_windows.append(f1_now)
    if baseline - f1_now > 0.08:
        drift_points.append(str(chunk["dt"].iloc[0].date()))
        online.partial_fit(frame_matrix(chunk), chunk["positive"])
        baseline = 0.7 * baseline + 0.3 * f1_score(chunk["positive"], online.predict(frame_matrix(chunk)), zero_division=0)
    start += window

online_future = online.predict(frame_matrix(df.iloc[cut_test:]))
online_f1 = f1_score(df.iloc[cut_test:]["positive"], online_future)
print(f"Пакетный SGD на будущем срезе, F1={batch_f1:.3f}")
print(f"Онлайн-SGD после дообучения на сдвигах, F1={online_f1:.3f}")
print("Окна, где F1 просело больше чем на 0.08:", drift_points)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(f1_windows) + 1), f1_windows, marker="o")
ax.set_title("F1 по временным окнам до дообучения окна")
ax.set_xlabel("Номер окна")
ax.set_ylabel("F1")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# PSI по оценкам: ранняя половина против поздней
def psi(left, right, bins=5):
    cuts = np.linspace(1, 5, bins + 1)
    e = np.histogram(left, bins=cuts)[0] / len(left)
    a = np.histogram(right, bins=cuts)[0] / len(right)
    e = np.clip(e, 1e-6, None)
    a = np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

half = len(df) // 2
rating_psi = psi(df["rating"].iloc[:half], df["rating"].iloc[half:])
print(f"PSI оценок между первой и второй половиной периода: {rating_psi:.3f}")
print(
    "Вывод по сдвигу: пакетная модель, застывшая на ранних отзывах, на поздних даёт F1 "
    f"{batch_f1:.3f}. Онлайн-обновление после просадки поднимает F1 до {online_f1:.3f}. "
    f"Срабатываний детектора: {len(drift_points)}. PSI оценок {rating_psi:.3f}: "
    "чем он дальше от нуля, тем сильнее поздние оценки отличаются от ранних."
)


Пакетный SGD на будущем срезе, F1=0.818
Онлайн-SGD после дообучения на сдвигах, F1=0.818
Окна, где F1 просело больше чем на 0.08: []
PSI оценок между первой и второй половиной периода: 0.011
Вывод по сдвигу: пакетная модель, застывшая на ранних отзывах, на поздних даёт F1 0.818. Онлайн-обновление после просадки поднимает F1 до 0.818. Срабатываний детектора: 0. PSI оценок 0.011: чем он дальше от нуля, тем сильнее поздние оценки отличаются от ранних.


  plt.show()


## Результаты


In [8]:
mlflow_note = (
    "Эксперименты записаны в MLflow. "
    if USING_MLFLOW
    else "Пакет MLflow в этой сессии не встал, метрики напечатаны в журнал запуска. "
)
print("Вывод")
print(
    f"На {len(df)} отзывах All_Beauty положительный класс — оценка 4 или 5. "
    f"Лучшая пакетная модель на позднем тесте: {best_name}, F1={score_table.iloc[0]['F1']:.3f}, "
    f"ROC-AUC={score_table.iloc[0]['ROC-AUC']:.3f}. {mlflow_note}"
    f"Признаки пользователя, посчитанные офлайн и тем же правилом онлайн, разошлись в {mismatch} строках. "
    f"p95 одиночного ответа {p95:.1f} мс при лимите 100 мс. "
    f"На будущем срезе застывший SGD дал F1 {batch_f1:.3f}, а модель с дообучением после просадок — {online_f1:.3f}. "
    "Для выдачи рекомендаций этого достаточно как быстрый текстовый признак; полный пересчёт по всему архиву остаётся ночной задачей, а сдвиг по времени закрывается онлайн-шагом."
)


Вывод
На 80000 отзывах All_Beauty положительный класс — оценка 4 или 5. Лучшая пакетная модель на позднем тесте: logreg, F1=0.895, ROC-AUC=0.914. Эксперименты записаны в MLflow. Признаки пользователя, посчитанные офлайн и тем же правилом онлайн, разошлись в 0 строках. p95 одиночного ответа 1.4 мс при лимите 100 мс. На будущем срезе застывший SGD дал F1 0.818, а модель с дообучением после просадок — 0.818. Для выдачи рекомендаций этого достаточно как быстрый текстовый признак; полный пересчёт по всему архиву остаётся ночной задачей, а сдвиг по времени закрывается онлайн-шагом.
